In [23]:
# Cell 1 — Install dependencies
!pip install requests -q
print("✅ Dependencies ready")

✅ Dependencies ready


In [24]:
# Cell 2 — Create a sample phishing .eml file for testing

sample_eml = """From: "PayPal Support" <support@paypa1-security.com>
To: victim@example.com
Subject: URGENT: Verify your account immediately or it will be suspended
Reply-To: attacker@gmail.com
Received: from mail.paypa1-security.com (192.168.1.50) by mx.example.com
Received: from unknown (185.220.101.45) by mail.paypa1-security.com
Date: Mon, 01 Jan 2024 10:00:00 +0000
MIME-Version: 1.0
Content-Type: text/plain

Dear Customer,

Your account has been compromised. Click here to verify your credentials
immediately or your account will be suspended within 24 hours.

Login here: http://paypa1-security.com/login

Thank you,
PayPal Security Team
"""

# Save it as a file in Colab's temporary storage
with open("sample.eml", "w") as f:
    f.write(sample_eml)

print("✅ sample.eml created")


✅ sample.eml created


In [25]:
# Cell 3 — Header Parser
import email

def load_email(file_path):
    """Opens and parses a .eml file."""
    with open(file_path, "r", encoding="utf-8") as f:
        return email.message_from_file(f)

def extract_headers(msg):
    """Extracts the key headers we care about."""
    return {
        "from":     msg.get("From", "Not found"),
        "to":       msg.get("To", "Not found"),
        "subject":  msg.get("Subject", "Not found"),
        "reply_to": msg.get("Reply-To", "Not found"),
        "date":     msg.get("Date", "Not found"),
        "received": msg.get_all("Received", []),
    }

def check_reply_to_mismatch(headers):
    """
    Checks if Reply-To domain differs from From domain.
    A classic phishing trick — attacker wants replies going somewhere else.
    """
    def get_domain(field):
        if "@" in field:
            return field.split("@")[-1].strip(">").strip().lower()
        return ""

    from_domain  = get_domain(headers["from"])
    reply_domain = get_domain(headers["reply_to"])

    if from_domain and reply_domain and from_domain != reply_domain:
        return True
    return False

def check_suspicious_display_name(headers):
    """
    Checks if the From display name contains alarm words.
    Scammers often write 'You've been HACKED' as the display
    name to cause panic before the victim even opens the email.
    """
    from_field = headers["from"].lower()

    display_name = ""
    if "<" in from_field:
        display_name = from_field.split("<")[0].strip().strip('"')

    alarm_words = [
        "hacked", "suspended", "compromised", "urgent", "warning",
        "alert", "blocked", "locked", "virus", "infected", "action required"
    ]

    found = [w for w in alarm_words if w in display_name]
    return found  # empty list = no alarm words found

print("✅ Header parser ready")

✅ Header parser ready


In [33]:
# Cell 4 — IP Extractor
import re

def is_valid_ip(ip):
    """
    Validates that:
    - Each octet is between 0-255
    - First octet is not 0 (e.g. 04.x.x.x is not a real routable IP)
    - No octet has a leading zero (04 is a date fragment, not an IP octet)
    """
    parts = ip.split(".")
    if len(parts) != 4:
        return False
    for part in parts:
        # Reject leading zeros — real IPs don't have them (04, 01, 02 etc.)
        if len(part) > 1 and part.startswith("0"):
            return False
        num = int(part)
        if not (0 <= num <= 255):
            return False
    # First octet can't be 0 — no real routable IP starts with 0
    if int(parts[0]) == 0:
        return False
    return True

def extract_ips(received_headers, date_header=""):
    """
    Pulls IPs from Received headers.
    Excludes anything that also appears in the Date header
    to avoid false positives like timestamps.
    """
    ip_pattern = r'\b(?:\d{1,3}\.){3}\d{1,3}\b'

    date_false_positives = re.findall(ip_pattern, date_header) if date_header else []

    found_ips = []
    for header in received_headers:
        for ip in re.findall(ip_pattern, header):
            if ip not in found_ips and is_valid_ip(ip) and ip not in date_false_positives:
                found_ips.append(ip)
    return found_ips

def is_private_ip(ip):
    """
    Private IPs (192.168.x.x, 10.x.x.x etc.) shouldn't appear
    in public email routing — flags it as suspicious.
    """
    private_ranges = [
        "10.", "127.", "192.168.",
        "172.16.", "172.17.", "172.18.", "172.19.",
        "172.20.", "172.21.", "172.22.", "172.23.",
        "172.24.", "172.25.", "172.26.", "172.27.",
        "172.28.", "172.29.", "172.30.", "172.31."
    ]
    return any(ip.startswith(r) for r in private_ranges)

def analyze_ips(ip_list):
    """Returns a list of dicts with analysis for each IP."""
    results = []
    for ip in ip_list:
        private = is_private_ip(ip)
        results.append({
            "ip": ip,
            "is_private": private,
            "note": "⚠️  Private IP in routing (suspicious)" if private else "Public IP"
        })
    return results

print("✅ IP extractor ready")

✅ IP extractor ready


In [27]:
# Cell 5 — Keyword Detector

def get_keyword_list():
    """Categorized phishing keywords."""
    return {
        "urgency": [
            "urgent", "immediately", "action required", "act now",
            "expires", "suspended", "24 hours", "last chance", "final notice"
        ],
        "fear": [
            "compromised", "unauthorized", "suspicious activity",
            "hacked", "breach", "locked", "disabled", "terminated"
        ],
        "credential_harvesting": [
            "verify", "confirm", "validate", "update your",
            "re-enter", "login", "sign in", "password", "credentials"
        ],
        "financial": [
            "bank account", "credit card", "payment", "billing",
            "invoice", "refund", "transaction", "wire transfer"
        ]
    }

def scan_text(text):
    """Scans text and returns matched keywords grouped by category."""
    text_lower = text.lower()
    findings = {}
    for category, keywords in get_keyword_list().items():
        matched = [kw for kw in keywords if kw in text_lower]
        if matched:
            findings[category] = matched
    return findings

def scan_subject_and_body(subject, body=""):
    return scan_text(subject + " " + body)

print("✅ Keyword detector ready")

✅ Keyword detector ready


In [28]:
# Cell 6 — Risk Scorer

def calculate_score(ip_analysis, keyword_findings, reply_to_mismatch, suspicious_display_name=None):
    """
    Scores the email from 0-100 based on all findings.
    Each indicator adds weighted points — higher = more suspicious.

    Weights:
      Suspicious display name : +30
      Reply-To mismatch       : +25
      Private IP in routing   : +15 each (max 30)
      Credential keywords     : +15
      Urgency keywords        : +10
      Fear keywords           : +10
      Financial keywords      : +10
    """
    score   = 0
    reasons = []

    if suspicious_display_name:
        score += 30
        reasons.append(f"Alarm words in sender display name: {', '.join(suspicious_display_name)} (+30)")

    if reply_to_mismatch:
        score += 25
        reasons.append("Reply-To domain differs from sender domain (+25)")

    private_ips = [r for r in ip_analysis if r["is_private"]]
    ip_points   = min(len(private_ips) * 15, 30)
    if private_ips:
        score += ip_points
        reasons.append(f"{len(private_ips)} private IP(s) in routing headers (+{ip_points})")

    keyword_weights = {
        "urgency":               10,
        "fear":                  10,
        "credential_harvesting": 15,
        "financial":             10,
    }
    for category, points in keyword_weights.items():
        if category in keyword_findings:
            score += points
            count = len(keyword_findings[category])
            reasons.append(f"{count} '{category}' keyword(s) detected (+{points})")

    return min(score, 100), reasons

def get_verdict(score):
    if score >= 70:
        return "🔴 HIGH RISK — Likely phishing"
    elif score >= 40:
        return "🟡 MEDIUM RISK — Suspicious, review carefully"
    elif score >= 15:
        return "🟠 LOW RISK — Some indicators present"
    else:
        return "🟢 CLEAN — No significant indicators found"

print("✅ Risk scorer ready")

✅ Risk scorer ready


In [29]:
# Cell 7 — Report Printer

def print_report(headers, ip_analysis, keyword_findings, score, reasons, verdict):
    print("\n" + "=" * 55)
    print("       PHISHING EMAIL ANALYSIS REPORT")
    print("=" * 55)

    print("\n📧 EMAIL HEADERS")
    print(f"  From:     {headers['from']}")
    print(f"  To:       {headers['to']}")
    print(f"  Subject:  {headers['subject']}")
    print(f"  Date:     {headers['date']}")
    print(f"  Reply-To: {headers['reply_to']}")

    print("\n🌐 IP ADDRESSES (from Received headers)")
    if ip_analysis:
        for entry in ip_analysis:
            print(f"  {entry['ip']}  —  {entry['note']}")
    else:
        print("  No IPs found in headers")

    print("\n🔍 KEYWORD ANALYSIS")
    if keyword_findings:
        for category, words in keyword_findings.items():
            print(f"  [{category.upper()}]  →  {', '.join(words)}")
    else:
        print("  No suspicious keywords detected")

    print("\n📊 RISK SCORE BREAKDOWN")
    if reasons:
        for reason in reasons:
            print(f"  • {reason}")
    else:
        print("  No risk factors found")

    print("\n" + "-" * 55)
    print(f"  RISK SCORE :  {score} / 100")
    print(f"  VERDICT    :  {verdict}")
    print("=" * 55 + "\n")

print("✅ Report printer ready")

✅ Report printer ready


In [34]:
# Cell 8 — RUN THE ANALYZER
# Change the filename below to analyze a different email
FILE_PATH = "Information about your online security (2).eml"

# Step 1: Load and parse the email file
msg     = load_email(FILE_PATH)
headers = extract_headers(msg)

# Step 2: Run header checks
reply_mismatch       = check_reply_to_mismatch(headers)
suspicious_disp_name = check_suspicious_display_name(headers)

# Step 3: Extract and analyze IPs (date_header filters false positives)
ips         = extract_ips(headers["received"], date_header=headers["date"])
ip_analysis = analyze_ips(ips)

# Step 4: Scan subject + body for phishing keywords
body             = msg.get_payload() if isinstance(msg.get_payload(), str) else ""
keyword_findings = scan_subject_and_body(headers["subject"], body)

# Step 5: Calculate risk score
score, reasons = calculate_score(
    ip_analysis,
    keyword_findings,
    reply_mismatch,
    suspicious_disp_name
)
verdict = get_verdict(score)

# Step 6: Print the report
print_report(headers, ip_analysis, keyword_findings, score, reasons, verdict)


       PHISHING EMAIL ANALYSIS REPORT

📧 EMAIL HEADERS
  From:     "You've been HACKED" <kfixc@kawachi.zaq.ne.jp>
  To:       you <you>
  Subject:  Information about your online security
  Date:     Sat, 4 Apr 2026 01:02:27 -0700
  Reply-To: Not found

🌐 IP ADDRESSES (from Received headers)
  222.227.81.164  —  Public IP

🔍 KEYWORD ANALYSIS
  [URGENCY]  →  immediately
  [FEAR]  →  hacked
  [FINANCIAL]  →  payment

📊 RISK SCORE BREAKDOWN
  • Alarm words in sender display name: hacked (+30)
  • 1 'urgency' keyword(s) detected (+10)
  • 1 'fear' keyword(s) detected (+10)
  • 1 'financial' keyword(s) detected (+10)

-------------------------------------------------------
  RISK SCORE :  60 / 100
  VERDICT    :  🟡 MEDIUM RISK — Suspicious, review carefully



In [36]:
# Cell 9 — Upload your own .eml file
from google.colab import files

uploaded = files.upload()  # a file picker will appear

# Get the uploaded filename
uploaded_file = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {uploaded_file}")
print("Now go to Cell 8, change FILE_PATH to:", f'"{uploaded_file}"', "and run it!")

Saving Information about your online security.eml to Information about your online security (3).eml

✅ Uploaded: Information about your online security (3).eml
Now go to Cell 8, change FILE_PATH to: "Information about your online security (3).eml" and run it!


In [38]:
# Cell 10 — Upload multiple .eml files
import os
from google.colab import files

# Create a folder to store all uploaded emails
os.makedirs("emails", exist_ok=True)

uploaded = files.upload()  # file picker — select multiple files at once

for filename, content in uploaded.items():
    with open(f"emails/{filename}", "wb") as f:
        f.write(content)

print(f"\n✅ {len(uploaded)} file(s) uploaded to /emails folder:")
for name in uploaded.keys():
    print(f"   • {name}")

Saving Information about your online security.eml to Information about your online security (4).eml

✅ 1 file(s) uploaded to /emails folder:
   • Information about your online security (4).eml


In [42]:
# Cell 11 — BATCH ANALYZER
# Analyzes every .eml file in the /emails folder
# and prints individual reports + a summary table at the end

import os

def analyze_single(file_path):
    """
    Runs the full analysis pipeline on one .eml file.
    Returns a dict of results for the summary table.
    """
    try:
        msg     = load_email(file_path)
        headers = extract_headers(msg)

        reply_mismatch       = check_reply_to_mismatch(headers)
        suspicious_disp_name = check_suspicious_display_name(headers)

        ips         = extract_ips(headers["received"], date_header=headers["date"])
        ip_analysis = analyze_ips(ips)

        body             = msg.get_payload() if isinstance(msg.get_payload(), str) else ""
        keyword_findings = scan_subject_and_body(headers["subject"], body)

        score, reasons = calculate_score(
            ip_analysis,
            keyword_findings,
            reply_mismatch,
            suspicious_disp_name
        )
        verdict = get_verdict(score)

        return {
            "file":    os.path.basename(file_path),
            "from":    headers["from"],
            "subject": headers["subject"],
            "score":   score,
            "verdict": verdict,
            "headers": headers,
            "ip_analysis": ip_analysis,
            "keyword_findings": keyword_findings,
            "reasons": reasons,
            "error":   None
        }

    except Exception as e:
        return {
            "file":    os.path.basename(file_path),
            "from":    "—",
            "subject": "—",
            "score":   0,
            "verdict": "⚪ ERROR",
            "error":   str(e)
        }


def print_summary_table(results):
    """
    Prints a ranked summary table of all analyzed emails.
    Sorted by risk score — highest first.
    """
    sorted_results = sorted(results, key=lambda x: x["score"], reverse=True)

    print("\n" + "=" * 75)
    print("                     BATCH ANALYSIS SUMMARY")
    print("=" * 75)
    print(f"  {'FILE':<25} {'SCORE':<8} {'VERDICT'}")
    print("-" * 75)

    for r in sorted_results:
        filename = r["file"][:24]  # truncate long filenames
        print(f"  {filename:<25} {r['score']:<8} {r['verdict']}")

    print("=" * 75)

    # Overall stats
    scores    = [r["score"] for r in results if r["error"] is None]
    high_risk = [r for r in results if r["score"] >= 70]

    if scores:
        print(f"\n  📁 Total emails analyzed : {len(results)}")
        print(f"  🔴 High risk emails      : {len(high_risk)}")
        print(f"  📊 Average risk score    : {sum(scores) // len(scores)} / 100")
        print(f"  📈 Highest score         : {max(scores)} / 100")
    print()


# ── Run the batch ──────────────────────────────────────────
email_folder = "emails"
eml_files    = [f for f in os.listdir(email_folder) if f.endswith(".eml")]

if not eml_files:
    print("⚠️  No .eml files found in /emails folder. Run Cell 10 to upload files.")
else:
    print(f"🔍 Found {len(eml_files)} email(s) — analyzing...\n")

    all_results = []

    for filename in eml_files:
        file_path = os.path.join(email_folder, filename)
        print(f"{'─' * 55}")
        print(f"  Analyzing: {filename}")
        print(f"{'─' * 55}")

        result = analyze_single(file_path)
        all_results.append(result)

        if result["error"]:
            print(f"  ❌ Error reading file: {result['error']}\n")
        else:
            # Print full individual report for each email
            print_report(
                result["headers"],
                result["ip_analysis"],
                result["keyword_findings"],
                result["score"],
                result["reasons"],
                result["verdict"]
            )

    # Print the summary table at the end
    print_summary_table(all_results)

🔍 Found 1 email(s) — analyzing...

───────────────────────────────────────────────────────
  Analyzing: Information about your online security (4).eml
───────────────────────────────────────────────────────

       PHISHING EMAIL ANALYSIS REPORT

📧 EMAIL HEADERS
  From:     "You've been HACKED" <kfixc@kawachi.zaq.ne.jp>
  To:       you <you>
  Subject:  Information about your online security
  Date:     Sat, 4 Apr 2026 01:02:27 -0700
  Reply-To: Not found

🌐 IP ADDRESSES (from Received headers)
  222.227.81.164  —  Public IP

🔍 KEYWORD ANALYSIS
  [URGENCY]  →  immediately
  [FEAR]  →  hacked
  [FINANCIAL]  →  payment

📊 RISK SCORE BREAKDOWN
  • Alarm words in sender display name: hacked (+30)
  • 1 'urgency' keyword(s) detected (+10)
  • 1 'fear' keyword(s) detected (+10)
  • 1 'financial' keyword(s) detected (+10)

-------------------------------------------------------
  RISK SCORE :  60 / 100
  VERDICT    :  🟡 MEDIUM RISK — Suspicious, review carefully


                     BATCH A

In [43]:
# Cell 12 — Export batch results to PDF
from google.colab import files

def export_to_pdf(results, filename="phishing_analysis_report.pdf"):
    """
    Exports batch analysis results to a clean, styled PDF report.
    """

    # Build the full HTML first — then convert to PDF using browser print
    html_rows = ""

    for r in sorted(results, key=lambda x: x["score"], reverse=True):

        # Verdict color
        score = r["score"]
        if score >= 70:
            verdict_color = "#e74c3c"   # red
            row_bg        = "#fdf0f0"
        elif score >= 40:
            verdict_color = "#e67e22"   # orange
            row_bg        = "#fef9f0"
        elif score >= 15:
            verdict_color = "#f39c12"   # yellow
            row_bg        = "#fffdf0"
        else:
            verdict_color = "#27ae60"   # green
            row_bg        = "#f0fdf4"

        # Pull data
        all_ips     = [e["ip"] for e in r["ip_analysis"]]
        private_ips = [e["ip"] for e in r["ip_analysis"] if e["is_private"]]
        kw          = r["keyword_findings"]

        urgency     = ", ".join(kw.get("urgency", []))               or "None"
        fear        = ", ".join(kw.get("fear", []))                  or "None"
        credential  = ", ".join(kw.get("credential_harvesting", [])) or "None"
        financial   = ", ".join(kw.get("financial", []))             or "None"

        # Score breakdown bullets
        breakdown_html = "<ul style='margin:4px 0; padding-left:18px;'>"
        for reason in r.get("reasons", []):
            breakdown_html += f"<li>{reason}</li>"
        if not r.get("reasons"):
            breakdown_html += "<li>No risk factors found</li>"
        breakdown_html += "</ul>"

        html_rows += f"""
        <div class="email-card" style="background:{row_bg};">
            <div class="card-header">
                <span class="filename">📄 {r['file']}</span>
                <span class="score-badge" style="background:{verdict_color};">
                    {score} / 100 — {r['verdict']}
                </span>
            </div>

            <table class="detail-table">
                <tr>
                    <td class="label">From</td>
                    <td>{r['headers']['from']}</td>
                    <td class="label">Date</td>
                    <td>{r['headers']['date']}</td>
                </tr>
                <tr>
                    <td class="label">Subject</td>
                    <td>{r['headers']['subject']}</td>
                    <td class="label">Reply-To</td>
                    <td>{r['headers']['reply_to']}</td>
                </tr>
                <tr>
                    <td class="label">IPs Found</td>
                    <td>{', '.join(all_ips) if all_ips else 'None'}</td>
                    <td class="label">Private IPs</td>
                    <td>{'⚠️ ' + ', '.join(private_ips) if private_ips else 'None'}</td>
                </tr>
            </table>

            <div class="section-title">🔍 Keywords Detected</div>
            <table class="detail-table">
                <tr>
                    <td class="label">Urgency</td>
                    <td>{urgency}</td>
                    <td class="label">Fear</td>
                    <td>{fear}</td>
                </tr>
                <tr>
                    <td class="label">Credential</td>
                    <td>{credential}</td>
                    <td class="label">Financial</td>
                    <td>{financial}</td>
                </tr>
            </table>

            <div class="section-title">📊 Risk Score Breakdown</div>
            {breakdown_html}
        </div>
        """

    # Summary stats
    scores    = [r["score"] for r in results if r.get("error") is None]
    high_risk = [r for r in results if r["score"] >= 70]
    avg_score = sum(scores) // len(scores) if scores else 0

    html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>Phishing Analysis Report</title>
        <style>
            * {{ box-sizing: border-box; margin: 0; padding: 0; }}

            body {{
                font-family: 'Segoe UI', Arial, sans-serif;
                font-size: 13px;
                color: #2c3e50;
                background: #fff;
                padding: 30px;
            }}

            /* ── Cover header ── */
            .report-header {{
                background: linear-gradient(135deg, #1a1a2e, #16213e);
                color: white;
                padding: 30px;
                border-radius: 10px;
                margin-bottom: 25px;
                text-align: center;
            }}
            .report-header h1 {{
                font-size: 26px;
                letter-spacing: 1px;
                margin-bottom: 6px;
            }}
            .report-header p {{
                color: #a0aec0;
                font-size: 13px;
            }}

            /* ── Summary bar ── */
            .summary-bar {{
                display: flex;
                gap: 15px;
                margin-bottom: 25px;
            }}
            .stat-box {{
                flex: 1;
                background: #f8f9fa;
                border: 1px solid #e2e8f0;
                border-radius: 8px;
                padding: 15px;
                text-align: center;
            }}
            .stat-box .stat-num {{
                font-size: 28px;
                font-weight: bold;
                color: #2c3e50;
            }}
            .stat-box .stat-label {{
                font-size: 11px;
                color: #718096;
                margin-top: 4px;
                text-transform: uppercase;
                letter-spacing: 0.5px;
            }}

            /* ── Email cards ── */
            .email-card {{
                border: 1px solid #e2e8f0;
                border-radius: 8px;
                padding: 18px;
                margin-bottom: 18px;
                page-break-inside: avoid;
            }}
            .card-header {{
                display: flex;
                justify-content: space-between;
                align-items: center;
                margin-bottom: 12px;
            }}
            .filename {{
                font-weight: bold;
                font-size: 14px;
                color: #2d3748;
            }}
            .score-badge {{
                color: white;
                padding: 5px 12px;
                border-radius: 20px;
                font-size: 12px;
                font-weight: bold;
            }}

            /* ── Tables ── */
            .detail-table {{
                width: 100%;
                border-collapse: collapse;
                margin-bottom: 10px;
                font-size: 12px;
            }}
            .detail-table td {{
                padding: 5px 8px;
                border: 1px solid #e2e8f0;
            }}
            .label {{
                background: #edf2f7;
                font-weight: bold;
                width: 12%;
                color: #4a5568;
                white-space: nowrap;
            }}

            /* ── Section titles ── */
            .section-title {{
                font-weight: bold;
                font-size: 12px;
                color: #4a5568;
                margin: 10px 0 5px 0;
                text-transform: uppercase;
                letter-spacing: 0.5px;
            }}

            ul li {{
                font-size: 12px;
                color: #4a5568;
                margin-bottom: 2px;
            }}

            /* ── Footer ── */
            .footer {{
                text-align: center;
                color: #a0aec0;
                font-size: 11px;
                margin-top: 30px;
                padding-top: 15px;
                border-top: 1px solid #e2e8f0;
            }}

            @media print {{
                body {{ padding: 15px; }}
                .email-card {{ page-break-inside: avoid; }}
            }}
        </style>
    </head>
    <body>

        <div class="report-header">
            <h1>🛡️ Phishing Email Analysis Report</h1>
            <p>Generated by Phishing Email Analyzer</p>
        </div>

        <div class="summary-bar">
            <div class="stat-box">
                <div class="stat-num">{len(results)}</div>
                <div class="stat-label">Emails Analyzed</div>
            </div>
            <div class="stat-box">
                <div class="stat-num" style="color:#e74c3c;">{len(high_risk)}</div>
                <div class="stat-label">High Risk</div>
            </div>
            <div class="stat-box">
                <div class="stat-num">{avg_score}</div>
                <div class="stat-label">Average Score</div>
            </div>
            <div class="stat-box">
                <div class="stat-num">{max(scores) if scores else 0}</div>
                <div class="stat-label">Highest Score</div>
            </div>
        </div>

        {html_rows}

        <div class="footer">
            Phishing Email Analyzer — For educational and research purposes only
        </div>

    </body>
    </html>
    """

    # Save the HTML file
    html_file = filename.replace(".pdf", ".html")
    with open(html_file, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"✅ Report generated!")
    print(f"   {len(results)} email(s) included\n")

    # Download the HTML — open in browser and Ctrl+P → Save as PDF
    files.download(html_file)
    print("📥 Downloaded as HTML.")
    print("👉 Open the file in your browser → press Ctrl+P → 'Save as PDF'")
    print("   This gives a perfectly formatted, print-ready PDF.")


# ── Run the export ─────────────────────────────────────────
try:
    export_to_pdf(all_results)
except NameError:
    print("⚠️  No results found. Please run Cell 11 (batch analyzer) first.")

✅ Report generated!
   1 email(s) included



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Downloaded as HTML.
👉 Open the file in your browser → press Ctrl+P → 'Save as PDF'
   This gives a perfectly formatted, print-ready PDF.
